In [ ]:
!pip install tensorflow keras opencv-python numpy matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install tensorflow keras opencv-python numpy


In [ ]:
%cd Emotion-recognition/models


[Errno 2] No such file or directory: 'Emotion-recognition/models'
/content


In [ ]:
%cd /content/Emotion-recognition/models


[Errno 2] No such file or directory: '/content/Emotion-recognition/models'
/content


In [ ]:
!git clone https://github.com/omar178/Emotion-recognition.git


Cloning into 'Emotion-recognition'...
remote: Enumerating objects: 219, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 219 (delta 4), reused 0 (delta 0), pack-reused 212 (from 4)
Receiving objects: 100% (219/219), 1.96 MiB | 3.76 MiB/s, done.
Resolving deltas: 100% (97/97), done.


In [ ]:
%cd /content/Emotion-recognition/models
!ls


/content/Emotion-recognition/models
cnn.py	_mini_XCEPTION.102-0.66.hdf5  __pycache__


In [ ]:
!df -h


Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   39G   75G  35% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G   56M  5.7G   1% /dev/shm
/dev/root       2.0G  1.2G  750M  62% /usr/sbin/docker-init
/dev/sda1        74G   41G   34G  55% /opt/bin/.nvidia
tmpfs           6.4G  240K  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
drive           113G   43G   71G  38% /content/drive


In [ ]:
!cp -r /content/drive/MyDrive/fer2013_dataset/fer2013 /content/fer2013


In [ ]:
import torch
print("GPU görüldü mü:", torch.cuda.is_available())
print("GPU adı:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK")


GPU görüldü mü: True
GPU adı: Tesla T4


In [ ]:
!ls /content/drive/MyDrive/fer2013_dataset/fer2013/train


angry  disgust	fear  happy  neutral  sad  surprise


In [ ]:
!ls /content
!ls /content/FER2013


In [ ]:
!ls /content/drive/MyDrive


In [ ]:
!ls /content/drive/MyDrive/fer2013_dataset


fer2013


In [ ]:
!ls /content/drive/MyDrive/fer2013_dataset/fer2013


test  train


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os
from tqdm import tqdm

# =========================
# 1. GPU
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Kullanılan cihaz:", device)

###
from torchvision.datasets import ImageFolder

class SafeImageFolder(ImageFolder):
    def __getitem__(self, index):
        img = None
        target = None

        # Bozuk görüntü çıkarsa atla
        while img is None:
            path, target = self.samples[index]
            img = safe_loader(path)

            if img is None:
                index = (index + 1) % len(self.samples)  # sonraki görüntüye geç

        if self.transform is not None:
            img = self.transform(img)

        return img, target

# =========================
# 2. DATA TRANSFORM
# =========================
data_transforms = {
    "train": transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.ToTensor(),
    ]),
    "test": transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]),
}

# =========================
# 3. DATALOADER
# =========================
from PIL import Image, UnidentifiedImageError

def safe_loader(path):
    try:
        return Image.open(path).convert('RGB')
    except UnidentifiedImageError:
        print(f"⚠ Bozuk dosya atlandı: {path}")
        return None

data_dir = "/content/fer2013"  # Kopyalanmış versiyon

image_datasets = {
    x: SafeImageFolder(os.path.join(data_dir, x), transform=data_transforms[x])
    for x in ["train", "test"]
}


dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=2)
    for x in ["train", "test"]
}

dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "test"]}
class_names = image_datasets["train"].classes

print("Train samples:", dataset_sizes["train"])
print("Test samples:", dataset_sizes["test"])
print("Classes:", class_names)

# =========================
# 4. MODEL (ResNet50)
# =========================
model = models.resnet50(weights="IMAGENET1K_V2")
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 7)  # 7 emotion classes
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

# =========================
# 5. EARLY STOPPING
# =========================
patience = 5
best_loss = float("inf")
counter = 0

# =========================
# 6. TRAIN + VALIDATION LOOP
# =========================
for epoch in range(50):
    print(f"\nEpoch {epoch+1}/50")

    # ------------------------
    # TRAIN
    # ------------------------
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in tqdm(dataloaders["train"]):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    train_loss = running_loss / dataset_sizes["train"]
    train_acc = running_corrects.double() / dataset_sizes["train"]

    # ------------------------
    # VALIDATION
    # ------------------------
    model.eval()
    val_loss = 0.0
    val_corrects = 0

    with torch.no_grad():
        for inputs, labels in dataloaders["test"]:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

    val_loss = val_loss / dataset_sizes["test"]
    val_acc = val_corrects.double() / dataset_sizes["test"]

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    # Scheduler validation loss'a göre çalışmalı
    scheduler.step(val_loss)

    # ------------------------
    # EARLY STOPPING
    # ------------------------
    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0

        save_path = "/content/drive/MyDrive/fer2013_resnet50_best.pth"
        torch.save(model.state_dict(), save_path)
        print("✔ Model kaydedildi:", save_path)
    else:
        counter += 1
        print(f"EarlyStopping Counter: {counter}/{patience}")

    if counter >= patience:
        print("⛔ Early stopping devreye girdi!")
        break

print("🎉 Eğitim tamamlandı!")


Kullanılan cihaz: cuda
Train samples: 28709
Test samples: 7178
Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

Epoch 1/50


100%|██████████| 898/898 [05:05<00:00,  2.94it/s]


⚠ Bozuk dosya atlandı: /content/fer2013/test/happy/PublicTest_86341575.jpg
Train Loss: 1.2541 | Train Acc: 0.5191
Val Loss:   1.0929 | Val Acc:   0.5903
✔ Model kaydedildi: /content/drive/MyDrive/fer2013_resnet50_best.pth

Epoch 2/50


100%|██████████| 898/898 [05:04<00:00,  2.95it/s]


⚠ Bozuk dosya atlandı: /content/fer2013/test/happy/PublicTest_86341575.jpg
Train Loss: 0.9809 | Train Acc: 0.6299
Val Loss:   0.9731 | Val Acc:   0.6317
✔ Model kaydedildi: /content/drive/MyDrive/fer2013_resnet50_best.pth

Epoch 3/50


...

⚠ Bozuk dosya atlandı: /content/fer2013/test/happy/PublicTest_86341575.jpg
Train Loss: 0.5180 | Train Acc: 0.8082
Val Loss:   1.0379 | Val Acc:   0.6790
EarlyStopping Counter: 3/5

Epoch 10/50


100%|██████████| 898/898 [05:04<00:00,  2.95it/s]


⚠ Bozuk dosya atlandı: /content/fer2013/test/happy/PublicTest_86341575.jpg
Train Loss: 0.3908 | Train Acc: 0.8590
Val Loss:   1.0088 | Val Acc:   0.6991
EarlyStopping Counter: 4/5

Epoch 11/50


100%|██████████| 898/898 [05:04<00:00,  2.95it/s]


⚠ Bozuk dosya atlandı: /content/fer2013/test/happy/PublicTest_86341575.jpg
Train Loss: 0.3293 | Train Acc: 0.8829
Val Loss:   1.0828 | Val Acc:   0.6995
EarlyStopping Counter: 5/5
⛔ Early stopping devreye girdi!
🎉 Eğitim tamamlandı!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

def get_resnet34_model(num_classes=7, fine_tune=True):
    # 1. ImageNet ağırlıklarıyla eğitilmiş ResNet34'ü yükle
    # Not: Yeni PyTorch sürümlerinde 'pretrained=True' yerine 'weights' parametresi önerilir.
    weights = models.ResNet34_Weights.DEFAULT
    model = models.resnet34(weights=weights)

    # 2. (Opsiyonel) Katmanları Dondurma (Freezing)
    # Eğer fine_tune=False ise, konvolüsyon katmanlarını dondururuz, sadece son katmanı eğitiriz.
    # FER2013 için genelde TÜM modeli eğitmek (fine_tune=True) daha iyi sonuç verir.
    if not fine_tune:
        for param in model.parameters():
            param.requires_grad = False

    # 3. Son Katmanı (Fully Connected) Değiştirme
    # ResNet'in orijinal son katmanı 1000 sınıf (ImageNet) içindir. Bunu 7 yapıyoruz.
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    return model

# Modeli oluştur ve GPU'ya at
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_resnet34_model(num_classes=7, fine_tune=True)
model = model.to(device)

print(f"Model {device} üzerinde çalışmaya hazır.")

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 136MB/s]


Model cuda üzerinde çalışmaya hazır.


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 32.3 MB/s eta 0:00:00


In [ ]:
# =========================================================
# 0. HYPERPARAMETERS
# =========================================================
config = {
    "batch_size": 64,
    "lr": 1e-4,
    "epochs": 50,
    "patience": 7,
    "img_size": 224,
    "model_name": "resnet34",
}

# =========================================================
# 1. IMPORTS
# =========================================================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from PIL import Image, UnidentifiedImageError
import os
from tqdm import tqdm

# =========================================================
# 2. GPU
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Kullanılan cihaz:", device)

# =========================================================
# 3. SAFE LOADER (Bozuk Dosya Atla)
# =========================================================
def safe_loader(path):
    try:
        return Image.open(path).convert('RGB')
    except UnidentifiedImageError:
        print(f"⚠ Bozuk dosya atlandı: {path}")
        return None

# =========================================================
# 4. SAFE IMAGEFOLDER
# =========================================================
from torchvision.datasets import ImageFolder

class SafeImageFolder(ImageFolder):
    def __getitem__(self, index):
        img = None
        target = None

        # Bozuk görüntü çıkarsa atla
        while img is None:
            path, target = self.samples[index]
            img = safe_loader(path)

            if img is None:
                index = (index + 1) % len(self.samples)  # sıradaki görsele geç

        if self.transform is not None:
            img = self.transform(img)

        return img, target

# =========================================================
# 5. DATA TRANSFORMS
# =========================================================
data_transforms = {
    "train": transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((config["img_size"], config["img_size"])),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.RandomResizedCrop(config["img_size"], scale=(0.8, 1.0)),
        transforms.ToTensor(),
    ]),
    "test": transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((config["img_size"], config["img_size"])),
        transforms.ToTensor(),
    ]),
}

# =========================================================
# 6. DATASET & DATALOADER
# =========================================================
data_dir = "/content/drive/MyDrive/fer2013_dataset/fer2013"
#data_dir = "/content/fer2013"

 # Kopyalanmış versiyon

image_datasets = {
    x: SafeImageFolder(os.path.join(data_dir, x), transform=data_transforms[x])
    for x in ["train", "test"]
}

dataloaders = {
    x: DataLoader(
        image_datasets[x],
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=2
    )
    for x in ["train", "test"]
}

dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "test"]}
class_names = image_datasets["train"].classes

print("Train samples:", dataset_sizes["train"])
print("Test samples:", dataset_sizes["test"])
print("Classes:", class_names)

# =========================================================
# 7. MODEL (ResNet34)
# =========================================================
model = models.resnet34(weights="IMAGENET1K_V1")

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 7)  # 7 emotion class
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config["lr"])
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

# =========================================================
# 8. EARLY STOPPING
# =========================================================
patience = config["patience"]
best_loss = float("inf")
counter = 0

# =========================================================
# 9. TRAINING LOOP
# =========================================================
for epoch in range(config["epochs"]):
    print(f"\nEpoch {epoch+1}/{config['epochs']}")

    # ------------------------
    # TRAIN
    # ------------------------
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in tqdm(dataloaders["train"]):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    train_loss = running_loss / dataset_sizes["train"]
    train_acc = running_corrects.double() / dataset_sizes["train"]

    # VALIDATION

    model.eval()
    val_loss = 0.0
    val_corrects = 0

    with torch.no_grad():
        for inputs, labels in dataloaders["test"]:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

    val_loss = val_loss / dataset_sizes["test"]
    val_acc = val_corrects.double() / dataset_sizes["test"]

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    # Scheduler validation'a bakar
    scheduler.step(val_loss)

    # ------------------------
    # EARLY STOPPING
    # ------------------------
    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0

        save_path = "/content/drive/MyDrive/fer2013_resnet34_best.pth"
        torch.save(model.state_dict(), save_path)
        print("✔ Model kaydedildi:", save_path)
    else:
        counter += 1
        print(f"EarlyStopping Counter: {counter}/{patience}")

    if counter >= patience:
        print("⛔ Early stopping devreye girdi!")
        break

print("🎉 Eğitim tamamlandı!")


Kullanılan cihaz: cuda
Train samples: 28709
Test samples: 7178
Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

Epoch 1/50


100%|██████████| 449/449 [03:44<00:00,  2.00it/s]


⚠ Bozuk dosya atlandı: /content/drive/MyDrive/fer2013_dataset/fer2013/test/happy/PublicTest_86341575.jpg
Train Loss: 1.1667 | Train Acc: 0.5579
Val Loss:   1.0570 | Val Acc:   0.6052
✔ Model kaydedildi: /content/drive/MyDrive/fer2013_resnet34_best.pth

Epoch 2/50


100%|██████████| 449/449 [03:04<00:00,  2.44it/s]


⚠ Bozuk dosya atlandı: /content/drive/MyDrive/fer2013_dataset/fer2013/test/happy/PublicTest_86341575.jpg
Train Loss: 0.9389 | Train Acc: 0.6467
Val Loss:   0.9626 | Val Acc:   0.6429
✔ Model kaydedildi: /content/drive/MyDrive/fer2013_resnet34_best.pth

Epoch 3/50


...

⚠ Bozuk dosya atlandı: /content/drive/MyDrive/fer2013_dataset/fer2013/test/happy/PublicTest_86341575.jpg
Train Loss: 0.2896 | Train Acc: 0.8971
Val Loss:   1.1032 | Val Acc:   0.7030
EarlyStopping Counter: 5/7

Epoch 13/50


100%|██████████| 449/449 [03:01<00:00,  2.47it/s]


⚠ Bozuk dosya atlandı: /content/drive/MyDrive/fer2013_dataset/fer2013/test/happy/PublicTest_86341575.jpg
Train Loss: 0.2562 | Train Acc: 0.9085
Val Loss:   1.1261 | Val Acc:   0.7017
EarlyStopping Counter: 6/7

Epoch 14/50


100%|██████████| 449/449 [03:02<00:00,  2.46it/s]


⚠ Bozuk dosya atlandı: /content/drive/MyDrive/fer2013_dataset/fer2013/test/happy/PublicTest_86341575.jpg
Train Loss: 0.1874 | Train Acc: 0.9353
Val Loss:   1.1643 | Val Acc:   0.7073
EarlyStopping Counter: 7/7
⛔ Early stopping devreye girdi!
🎉 Eğitim tamamlandı!


In [ ]:

#Drive’daki FER2013 veri setini kopyalayıp Colab'ın kendi diskine aktarmak için!mkdir -p /content/fer2013/train
!mkdir -p /content/fer2013/test

!cp -r "/content/drive/MyDrive/fer2013_dataset/fer2013/train" /content/fer2013/
!cp -r "/content/drive/MyDrive/fer2013_dataset/fer2013/test" /content/fer2013/


In [ ]:
!ls "/content/drive/MyDrive/archive (15)/train/angry" | head


In [ ]:
import os
import time
import copy
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm  # <--- TQDM EKLENDİ
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets
from torch.cuda.amp import autocast, GradScaler
import timm

# ==========================================
# 1. HİPERPARAMETRELER
# ==========================================
class Config:
    # Veri seti yolu (Klasör yapısı: train/angry, test/happy vb.)
    #data_dir = "/content/drive/MyDrive/fer2013_dataset/fer2013"
    data_dir = "/content/fer2013"

    model_name = 'vit_base_patch16_224'
    num_classes = 7
    batch_size = 64
    num_epochs = 50
    learning_rate = 1e-4
    weight_decay = 1e-4
    patience = 5
    seed = 42
    use_amp = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint_path = 'best_vit_fer2013.pth'

def seed_everything(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

seed_everything(Config.seed)

#clasifiction reposts
def classification_report_eval(model, loader, config, class_names):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Classification Report"):
            images = images.to(config.device)
            labels = labels.to(config.device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n📊 Classification Report:\n")
    print(classification_report(
        all_labels,
        all_preds,
        target_names=class_names
    ))

# ==========================================
# 2. DATASET VE DATALOADER
# ==========================================
def get_dataloaders(config):
    train_dir = os.path.join(config.data_dir, 'train')
    test_dir = os.path.join(config.data_dir, 'test')

    if not os.path.exists(train_dir):
        # Eğer klasör yoksa hata fırlatmadan önce kullanıcıyı uyaralım
        print(f"UYARI: {train_dir} bulunamadı. Lütfen veri setini doğru klasöre çıkardığından emin ol.")
        # Test amaçlı dummy data dönmemesi için burada raise edebiliriz veya
        # kodun çalışıp hatayı main'de yakalamasını bekleyebiliriz.
        raise FileNotFoundError(f"{train_dir} bulunamadı.")

    # ViT için 224x224 boyutlandırma ve normalizasyon
    train_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
    val_dataset = datasets.ImageFolder(root=test_dir, transform=val_transforms)

    print(f"Sınıflar: {train_dataset.classes}")

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=4, pin_memory=True)

    return train_loader, val_loader


# ==========================================
# 3. MODEL
# ==========================================
def build_model(config):
    model = timm.create_model(config.model_name, pretrained=True, num_classes=config.num_classes)
    model.to(config.device)
    return model

# ==========================================
# 4. TRAIN (TQDM Eklendi)
# ==========================================
def train_one_epoch(model, loader, criterion, optimizer, scaler, config):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # TQDM Bar Tanımlaması
    pbar = tqdm(loader, desc="Training", leave=False)

    for images, labels in pbar:
        images, labels = images.to(config.device), labels.to(config.device)

        optimizer.zero_grad()

        # AMP Aktif ise
        if config.use_amp:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # İstatistikler
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Anlık durumu progress bar'a yaz
        current_loss = running_loss / total
        current_acc = 100. * correct / total
        pbar.set_postfix({"loss": f"{current_loss:.4f}", "acc": f"{current_acc:.2f}%"})

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

# ==========================================
# 5. EVAL (TQDM Eklendi)
# ==========================================
def evaluate(model, loader, criterion, config):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    # TQDM Bar Tanımlaması
    pbar = tqdm(loader, desc="Validating", leave=False)

    with torch.no_grad():
        for images, labels in pbar:
            images, labels = images.to(config.device), labels.to(config.device)

            if config.use_amp:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

            # İstatistikler
            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            # Anlık durumu progress bar'a yaz
            current_loss = running_loss / total
            current_acc = 100. * correct / total
            pbar.set_postfix({"val_loss": f"{current_loss:.4f}", "val_acc": f"{current_acc:.2f}%"})

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

# ==========================================
# 6. MAIN
# ==========================================
def main():
    print(f"Model: {Config.model_name}")
    print(f"Cihaz: {Config.device}")

    # 1. Veri Yükle
    try:
        train_loader, val_loader = get_dataloaders(Config)
    except Exception as e:
        print(f"Hata: {e}")
        return
    class_names = train_loader.dataset.classes
    print("Sınıf İsimleri:", class_names)
    # 2. Modeli Kur
    model = build_model(Config)

    # 3. Optimizer ve Loss
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=Config.learning_rate, weight_decay=Config.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.num_epochs)
    scaler = GradScaler(enabled=Config.use_amp)

    #best_acc = 0.0
    patience_counter = 0
    best_loss = float('inf')

    print("Eğitim Başlıyor...")
    print("-" * 60)

    for epoch in range(Config.num_epochs):
        start_time = time.time()

        # Train Step
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, Config)

        # Eval Step
        val_loss, val_acc = evaluate(model, val_loader, criterion, Config)

        scheduler.step()

        elapsed = time.time() - start_time

        # Epoch Özeti (Terminalde kalıcı olan satır burası)
        print(f"Epoch [{epoch+1}/{Config.num_epochs}] "
              f"Time: {elapsed:.0f}s | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")

# Checkpoint (LOSS'a göre kontrol)
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), Config.checkpoint_path)
            # DÜZELTME: Acc yerine Loss yazdık ve % işaretini kaldırdık.
            print(f"--> En iyi model kaydedildi! (Loss: {best_loss:.4f})")
        else:
            patience_counter += 1
            print(f"--> İyileşme yok. Sabır: {patience_counter}/{Config.patience}")

        if patience_counter >= Config.patience:
            print("Early Stopping tetiklendi. Eğitim durduruluyor.")
            break

    print("-" * 60)
    # DÜZELTME: Bitiş mesajını da Loss'a uygun hale getirdik.
    print(f"Eğitim tamamlandı. En Düşük Validasyon Kaybı (Best Loss): {best_loss:.4f}")
# ==============================
    # CLASSIFICATION REPORT
    # ==============================
    print("\nEn iyi model yükleniyor...")
    model.load_state_dict(torch.load(Config.checkpoint_path))

    classification_report_eval(
        model=model,
        loader=val_loader,
        config=Config,
        class_names=class_names
    )
if __name__ == "__main__":
    main()


Model: vit_base_patch16_224
Cihaz: cuda
Sınıflar: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Sınıf İsimleri: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


/tmp/ipython-input-3244239341.py:229: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=Config.use_amp)


Eğitim Başlıyor...
------------------------------------------------------------


Training:   0%|          | 0/449 [00:00<?, ?it/s]/tmp/ipython-input-3244239341.py:138: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validating:   0%|          | 0/113 [00:00<?, ?it/s]/tmp/ipython-input-3244239341.py:184: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/50] Time: 247s | Train Loss: 1.1385 Acc: 57.03% | Val Loss: 0.9538 Acc: 63.51%
--> En iyi model kaydedildi! (Loss: 0.9538)


...

Epoch [8/50] Time: 245s | Train Loss: 0.4550 Acc: 83.49% | Val Loss: 0.9542 Acc: 69.64%
--> İyileşme yok. Sabır: 3/5


Epoch [9/50] Time: 246s | Train Loss: 0.3895 Acc: 85.67% | Val Loss: 1.0382 Acc: 68.72%
--> İyileşme yok. Sabır: 4/5


Epoch [10/50] Time: 245s | Train Loss: 0.3370 Acc: 88.00% | Val Loss: 1.0800 Acc: 69.29%
--> İyileşme yok. Sabır: 5/5
Early Stopping tetiklendi. Eğitim durduruluyor.
------------------------------------------------------------
Eğitim tamamlandı. En Düşük Validasyon Kaybı (Best Loss): 0.8788

En iyi model yükleniyor...


Classification Report: 100%|██████████| 113/113 [01:18<00:00,  1.45it/s]



📊 Classification Report:

              precision    recall  f1-score   support

       angry       0.61      0.51      0.55       958
     disgust       0.66      0.64      0.65       111
        fear       0.49      0.56      0.52      1024
       happy       0.89      0.87      0.88      1773
     neutral       0.61      0.71      0.65      1233
         sad       0.61      0.50      0.55      1247
    surprise       0.75      0.81      0.78       831

    accuracy                           0.67      7177
   macro avg       0.66      0.66      0.66      7177
weighted avg       0.68      0.67      0.67      7177



In [ ]:
import os
import time
import copy
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm  # <--- TQDM EKLENDİ
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets
from torch.cuda.amp import autocast, GradScaler
import timm

# ==========================================
# 1. HİPERPARAMETRELER
# ==========================================
class Config:
    # Veri seti yolu (Klasör yapısı: train/angry, test/happy vb.)
    #data_dir = "/content/drive/MyDrive/fer2013_dataset/fer2013"
    data_dir = "/content/fer2013"

    model_name = 'vit_base_patch16_224'
    num_classes = 7
    batch_size = 64
    num_epochs = 50
    learning_rate = 1e-4
    weight_decay = 1e-4
    patience = 5
    seed = 42
    use_amp = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint_path = 'best_vit_fer2013.pth'

def seed_everything(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

seed_everything(Config.seed)

# ==========================================
# 2. DATASET VE DATALOADER
# ==========================================
def get_dataloaders(config):
    train_dir = os.path.join(config.data_dir, 'train')
    test_dir = os.path.join(config.data_dir, 'test')


    if not os.path.exists(train_dir):
        # Eğer klasör yoksa hata fırlatmadan önce kullanıcıyı uyaralım
        print(f"UYARI: {train_dir} bulunamadı. Lütfen veri setini doğru klasöre çıkardığından emin ol.")
        # Test amaçlı dummy data dönmemesi için burada raise edebiliriz veya
        # kodun çalışıp hatayı main'de yakalamasını bekleyebiliriz.
        raise FileNotFoundError(f"{train_dir} bulunamadı.")

    train_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
    val_dataset = datasets.ImageFolder(root=test_dir, transform=val_transforms)

    print(f"Sınıflar: {train_dataset.classes}")

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=4, pin_memory=True)

    return train_loader, val_loader

# ==========================================
# 3. MODEL
# ==========================================
def build_model(config):
    model = timm.create_model(config.model_name, pretrained=True, num_classes=config.num_classes)
    model.to(config.device)
    return model

# ==========================================
# 4. TRAIN (TQDM Eklendi)
# ==========================================
def train_one_epoch(model, loader, criterion, optimizer, scaler, config):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # TQDM Bar Tanımlaması
    pbar = tqdm(loader, desc="Training", leave=False)

    for images, labels in pbar:
        images, labels = images.to(config.device), labels.to(config.device)

        optimizer.zero_grad()

        # AMP Aktif ise
        if config.use_amp:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # İstatistikler
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Anlık durumu progress bar'a yaz
        current_loss = running_loss / total
        current_acc = 100. * correct / total
        pbar.set_postfix({"loss": f"{current_loss:.4f}", "acc": f"{current_acc:.2f}%"})

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

# ==========================================
# 5. EVAL (TQDM Eklendi)
# ==========================================
def evaluate(model, loader, criterion, config):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    # TQDM Bar Tanımlaması
    pbar = tqdm(loader, desc="Validating", leave=False)

    with torch.no_grad():
        for images, labels in pbar:
            images, labels = images.to(config.device), labels.to(config.device)

            if config.use_amp:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

            # İstatistikler
            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            # Anlık durumu progress bar'a yaz
            current_loss = running_loss / total
            current_acc = 100. * correct / total
            pbar.set_postfix({"val_loss": f"{current_loss:.4f}", "val_acc": f"{current_acc:.2f}%"})

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

# ==========================================
# 6. MAIN
# ==========================================
def main():
    print(f"Model: {Config.model_name}")
    print(f"Cihaz: {Config.device}")

    # 1. Veri Yükle
    try:
        train_loader, val_loader = get_dataloaders(Config)
    except Exception as e:
        print(f"Hata: {e}")
        return
    class_names = train_loader.dataset.classes
    print("Sınıf İsimleri:", class_names)
    # 2. Modeli Kur
    model = build_model(Config)

    # 3. Optimizer ve Loss
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=Config.learning_rate, weight_decay=Config.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.num_epochs)
    scaler = GradScaler(enabled=Config.use_amp)

    #best_acc = 0.0
    patience_counter = 0
    best_loss = float('inf')

    print("Eğitim Başlıyor...")
    print("-" * 60)

    for epoch in range(Config.num_epochs):
        start_time = time.time()

        # Train Step
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, Config)

        # Eval Step
        val_loss, val_acc = evaluate(model, val_loader, criterion, Config)

        scheduler.step()

        elapsed = time.time() - start_time

        # Epoch Özeti (Terminalde kalıcı olan satır burası)
        print(f"Epoch [{epoch+1}/{Config.num_epochs}] "
              f"Time: {elapsed:.0f}s | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")

# Checkpoint (LOSS'a göre kontrol)
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), Config.checkpoint_path)
            # DÜZELTME: Acc yerine Loss yazdık ve % işaretini kaldırdık.
            print(f"--> En iyi model kaydedildi! (Loss: {best_loss:.4f})")
        else:
            patience_counter += 1
            print(f"--> İyileşme yok. Sabır: {patience_counter}/{Config.patience}")

        if patience_counter >= Config.patience:
            print("Early Stopping tetiklendi. Eğitim durduruluyor.")
            break

    print("-" * 60)
    # DÜZELTME: Bitiş mesajını da Loss'a uygun hale getirdik.
    print(f"Eğitim tamamlandı. En Düşük Validasyon Kaybı (Best Loss): {best_loss:.4f}")
# ==============================
    # CLASSIFICATION REPORT
    # ==============================
    print("\nEn iyi model yükleniyor...")
    model.load_state_dict(torch.load(Config.checkpoint_path))

    classification_report_eval(
        model=model,
        loader=val_loader,
        config=Config,
        class_names=class_names
    )
if __name__ == "__main__":
    main()

Model: vit_base_patch16_224
Cihaz: cuda
Sınıflar: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Sınıf İsimleri: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/tmp/ipython-input-2747970662.py:206: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=Config.use_amp)


Eğitim Başlıyor...
------------------------------------------------------------


Training:   0%|          | 0/449 [00:00<?, ?it/s]/tmp/ipython-input-2747970662.py:115: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validating:   0%|          | 0/113 [00:00<?, ?it/s]/tmp/ipython-input-2747970662.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/50] Time: 251s | Train Loss: 1.3111 Acc: 48.97% | Val Loss: 1.0164 Acc: 60.90%
--> En iyi model kaydedildi! (Loss: 1.0164)


...

Epoch [8/50] Time: 249s | Train Loss: 0.5624 Acc: 79.45% | Val Loss: 0.8602 Acc: 69.68%
--> İyileşme yok. Sabır: 3/5


Epoch [9/50] Time: 248s | Train Loss: 0.5004 Acc: 81.59% | Val Loss: 0.9042 Acc: 69.69%
--> İyileşme yok. Sabır: 4/5


Epoch [10/50] Time: 248s | Train Loss: 0.4443 Acc: 83.74% | Val Loss: 0.9611 Acc: 69.43%
--> İyileşme yok. Sabır: 5/5
Early Stopping tetiklendi. Eğitim durduruluyor.
------------------------------------------------------------
Eğitim tamamlandı. En Düşük Validasyon Kaybı (Best Loss): 0.8550

En iyi model yükleniyor...


Classification Report: 100%|██████████| 113/113 [01:17<00:00,  1.45it/s]



📊 Classification Report:

              precision    recall  f1-score   support

       angry       0.68      0.54      0.60       958
     disgust       0.70      0.56      0.62       111
        fear       0.58      0.42      0.48      1024
       happy       0.89      0.88      0.88      1773
     neutral       0.63      0.74      0.68      1233
         sad       0.53      0.62      0.57      1247
    surprise       0.75      0.85      0.80       831

    accuracy                           0.69      7177
   macro avg       0.68      0.66      0.66      7177
weighted avg       0.69      0.69      0.69      7177



In [ ]:
from PIL import Image
import os

def clean_corrupted_images(root_dir):
    removed = 0
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                with Image.open(file_path) as img:
                    img.verify()  # resim mi kontrol et
            except Exception:
                print("Siliniyor:", file_path)
                os.remove(file_path)
                removed += 1
    print(f"✅ Temizleme tamamlandı. Silinen dosya sayısı: {removed}")

# TRAIN ve TEST için temizle
clean_corrupted_images("/content/fer2013/train")
clean_corrupted_images("/content/fer2013/test")


✅ Temizleme tamamlandı. Silinen dosya sayısı: 0
Siliniyor: /content/fer2013/test/happy/PublicTest_86341575.jpg
✅ Temizleme tamamlandı. Silinen dosya sayısı: 1


In [ ]:
!ls -R /content/fer2013_dataset_local


ls: cannot access '/content/fer2013_dataset_local': No such file or directory


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
Tesla T4


In [ ]:
!cp -r /content/drive/MyDrive/fer2013_dataset /content/fer2013_dataset_local
